# 🧠 Alignment with similarity and optimization

**Xu Lab | Biology-to-CryoEM Research | Module 10**  
**Phase:** C: Classical cryo-ET discovery | **Level:** scaffolded undergraduate → research | **Time:** 60–100 minutes  
**Hardware:** Google Colab free-tier CPU (optional T4 GPU for later extensions)  
**Core tools:** Python, NumPy, SciPy, Matplotlib; selected modules use scikit-learn/PyTorch  
**Prerequisites:** Previous tutorial  
**Status:** self-contained *pedagogical demonstration* using synthetic data, not an official reproduction of Xu Lab publications.  
**Safety:** for research and educational purposes only, not for clinical diagnosis.

## Learning objectives

1. Explain the biological question and physical limitations.
2. Derive or interpret the model below and identify its assumptions.
3. Run the experiment, change a parameter, and interpret quantitative output.
4. Explain how the concept supports the research papers linked below.

## Biological motivation

Particles must be brought into the same coordinate frame before structural averaging.

## Conceptual bridge

Estimate 2D translation from cross-correlation; then explore 3D rotation with a coarse orientation grid. This is a bridge to the existing STA notebook.

## Mathematical model

$(\widehat R,\widehat{\mathbf t})=\arg\max_{R,\mathbf t}\operatorname{NCC}(V_1,T_{R,\mathbf t}V_2)$.

## Notebook roadmap

1. Reproducible setup  2. Synthetic experiment  3. Interpretation  4. Challenge  5. Paper connection.


In [ ]:
# -- Environment check and shared utilities --
import sys, importlib.util
import numpy as np
import scipy
from scipy import ndimage as ndi, signal
import matplotlib.pyplot as plt
SEED=2026
rng=np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi':95,'axes.grid':False})
print('Python:',sys.version.split()[0], 'NumPy:',np.__version__)
print('PyTorch available:',bool(importlib.util.find_spec('torch')))
try:
    import torch
    print('CUDA available:',torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch not installed; required only for Module 13/14')
# Colab stock environment normally contains the required libraries.
# If an import fails, run: %pip install -q numpy scipy matplotlib scikit-learn torch

def phantom(n=48):
    """Build a deterministic 3D synthetic density with three distinct macromolecular blobs."""
    zz, yy, xx = np.mgrid[:n,:n,:n].astype(float)
    rho = np.zeros((n,n,n),dtype=float)
    for z,y,x,s,amp in [(0.45*n,.47*n,.48*n,.10*n,1.),(.65*n,.38*n,.58*n,.07*n,.7),(.40*n,.66*n,.36*n,.075*n,.8)]:
        rho += amp*np.exp(-((zz-z)**2+(yy-y)**2+(xx-x)**2)/(2*s*s))
    return rho/np.max(rho)


## Experiment: build, inspect, and check a scientific claim

Run the code below, then identify (i) input, (ii) transformation, (iii) output and (iv) a key assumption.

In [ ]:
im=phantom(64)[32]; shifted=ndi.shift(im,shift=(6,-9),order=1,mode='constant')
c=signal.fftconvolve(im,shifted[::-1,::-1],mode='full')
yx=np.unravel_index(np.argmax(c),c.shape); displacement=np.array(yx)-np.array(im.shape)+1
print('Shift to align image 2 to image 1:',displacement)
assert np.allclose(displacement,[-6,9],atol=1)
fixed=ndi.shift(shifted,shift=displacement,order=1)
print('MSE before/after:',np.mean((im-shifted)**2),np.mean((im-fixed)**2))

## Explain what happened

**Expected scientific observation:** Rigid alignment requires 3 rotations and 3 translations in 3D; missing-wedge support should be accounted for in similarity.

Write 3–5 sentences connecting the plotted result to the biological motivation. Describe one failure case and one alternative explanation.

## Guided challenge (edit code above)

Add a 15° rotation before translation. Explain why pure correlation may then fail and why a missing-wedge-aware distance is useful.

**Deliverable:** a labeled figure or small results table, a claim, and one limitation. Quantify the effect rather than just saying “it looks better.”

## Research connection and reading strategy

**Related Xu Lab research:** 2012 fast matching; 2012 parallel gradient alignment; 2018 ML averaging; 2020 Gum-Net.

Read each paper in this order: biological question → observations/data → forward model → key objective → baseline → evaluation split → limitations. Distinguish what the toy exercise illustrates from the actual published implementation. See `../PAPER_TO_TUTORIAL_MAP.md` for publication titles and further prerequisites.

## Exit ticket

1. What is known from the measured data, versus assumed by the model?
2. Which plotted quantity could be misleading if evaluated only on synthetic data?
3. Sketch one controlled ablation and one biological validation experiment.
4. Save an updated notebook and a 150-word lab note.